In [ ]:
!pip install datasets, pillow

In [ ]:
import os

access_token = ""

if os.path.exists("hf_token"):
    with open("hf_token", "r") as f:
        access_token = f.read().strip()

os.environ["HF_TOKEN"] = access_token

In [ ]:
import io

import zipfile
from PIL import Image
from huggingface_hub import HfFileSystem

# Initialize the file system
fs = HfFileSystem()


pubmed_filenames = [
    "00_39_PMC10381143_jimaging-09-00148-g011_1.jpg",
    "00_39_PMC10381143_jimaging-09-00148-g012_1.jpg",
    "0d_59_PMC4458964_IJD_60_321e_g003_0.png"
]

IYII_filenames = [
    "2281_1.png"
]

youtube_filenames = [
    "OzBKmWt9zWo_frame_8071_0_0.jpg"
]

repo_id = "datasets/redlessone/Derm1M"
downloaded_images = {}

print("Fetching individual images via streamed remote zipfile...")

download_tasks = {
    # "pubmed": pubmed_filenames,
    # "IIYI": IYII_filenames,
    "youtube": youtube_filenames
}

for category, filenames in download_tasks.items():
    zip_name = f"{category}.zip"
    remote_zip_path = f"{repo_id}/{zip_name}"
    
    try:
        with fs.open(remote_zip_path, "rb") as remote_file:
            with zipfile.ZipFile(remote_file) as zf:
                
                for filename in filenames:
                    internal_path = f"{filename}"
                    
                    try:
                        with zf.open(internal_path) as img_file:
                            img = Image.open(io.BytesIO(img_file.read())).convert("RGB")
                            
                            downloaded_images[filename] = img
                            
                            local_save_path = f"derm_train_images/{category}/{filename}"
                            os.makedirs(os.path.dirname(local_save_path), exist_ok=True)
                            img.save(local_save_path)
                            
                            print(f"[SUCCESS] Downloaded: {filename} to {local_save_path}")
                            
                    except KeyError:
                        print(f"[ERROR] File '{internal_path}' does not exist inside '{zip_name}'.")
                    except Exception as e:
                        print(f"[ERROR] Failed to download {internal_path}: {e}")
                        
    except Exception as e:
        print(f"[ERROR] Could not open remote zip '{remote_zip_path}': {e}")

if downloaded_images:
    first_image = list(downloaded_images.values())[0]
    display(first_image)

In [ ]:
from datasets import load_dataset

ds = load_dataset("joshuachou/SkinCAP", streaming=True)

sample = ds["train"].take(5) 
os.makedirs("SkinCAP_images", exist_ok=True)

for i, item in enumerate(list(sample)):
    item["image"].save(f"SkinCAP_images/{i+1}.png")

In [ ]:
meta  = load_dataset(
        "joshuachou/SkinCAP",
        data_files="skincap_v240623.csv"
    )

df = meta["train"].to_pandas()
df = df.head(5)
df["caption"] = df["caption_zh_polish_en"]
df["label"] = df["disease"]
df["filepath"] = df["skincap_file_path"]
df.to_csv("SkinCAP_images/meta.csv", index=False)

# To download metadata into the dict:
# meta_df = pd.read_csv("SkinCAP_images/meta.csv")
# meta_dict = meta_df.to_dict(orient="records")
    

In [ ]:
# DO NOT RUN AGAIN

!curl -u researcher:p7skn!dkah -O "https://derm.cs.sfu.ca/restricted/release_v0.zip" && tar -xf release_v0.zip

In [ ]:
# DO NOT RUN AGAIN
import shutil
import pandas as pd

out_dir = "derm7pt_images/"
os.makedirs(out_dir, exist_ok=True)
in_dir_meta = "release_v0/meta/meta.csv"
in_dir_images = "release_v0/images/"
df = pd.read_csv(in_dir_meta)
sample = df.sample(5)

for index, row in sample.iterrows():
    shutil.copy(in_dir_images + row["derm"], out_dir)
    
sample = sample.rename(columns={"diagnosis": "label"})
# sample["caption"] = sample["diagnosis"]
sample["filepath"] = sample["derm"].apply(lambda x: os.path.join(out_dir, os.path.basename(x)))
sample.to_csv(out_dir + "meta.csv", index=False)
shutil.rmtree("release_v0")
os.remove("release_v0.zip")
# To download metadata into the dict:
# meta_df = pd.read_csv("derm7pt_images/meta.csv")
# meta_dict = meta_df.to_dict(orient="records")